In [1]:
import sys
print(sys.executable)

c:\Users\david\GitHub\TESI\.venv\Scripts\python.exe


In [3]:
# =============================
# SETUP
# =============================
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (c:\Users\david\GitHub\TESI\.venv\Lib\site-packages\huggingface_hub\__init__.py)

In [ ]:
# =============================
# CONFIG
# =============================
DATASET = "small"
BASE = Path(f"data/processed/{DATASET}")

INPUT_FILE = BASE / "movies_enriched_tmdb.csv"

EMBED_FILE = BASE / "movie_embeddings_v2.npy"
INDEX_FILE = BASE / "movie_embeddings_index_v2.csv"
SIM_FILE = BASE / "content_similarity_matrix_v2.npy"

TRAIN_FILE = BASE / "ratings_train.csv"
VAL_FILE = BASE / "ratings_val.csv"
TEST_FILE = BASE / "ratings_test.csv"

In [ ]:
# =============================
# LOAD DATA
# =============================
df = pd.read_csv(INPUT_FILE)
print("Movies loaded:", df.shape)
df.head()

In [ ]:
# =============================
# TEXT PREPROCESSING
# =============================

def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def normalize_name_list(value):
    if pd.isna(value):
        return ""
    items = []
    for x in str(value).split(","):
        x = x.strip().lower()
        if x:
            items.append(x.replace(" ", "_"))
    return " ".join(items)

def normalize_genres(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower().replace("|", " ")

def build_text_representation(row):
    genres = normalize_genres(row.get("genres", ""))
    actors = normalize_name_list(row.get("actors_top5", ""))
    director = normalize_name_list(row.get("director", ""))
    overview = normalize_text(row.get("overview_en", ""))

    text = (
        f"genres {genres}. "
        f"genres {genres}. "
        f"genres {genres}. "
        f"director {director}. "
        f"director {director}. "
        f"actors {actors}. "
        f"actors {actors}. "
        f"plot {overview}"
    )

    return text.strip()

df["text_repr_v2"] = df.apply(build_text_representation, axis=1)

df[["title_clean", "text_repr_v2"]].head()

In [ ]:
# =============================
# EMBEDDINGS
# =============================
model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(
    df["text_repr_v2"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

np.save(EMBED_FILE, embeddings)

df_index = df[["movieId", "title_clean"]]
df_index.to_csv(INDEX_FILE, index=False)

print("Saved embeddings + index")

In [ ]:
# =============================
# SIMILARITY MATRIX
# =============================
sim_matrix = cosine_similarity(embeddings)

print("Similarity shape:", sim_matrix.shape)

np.save(SIM_FILE, sim_matrix)
print("Saved similarity matrix")

In [ ]:
# =============================
# RECOMMENDER
# =============================
class ContentRecommender:
    def __init__(self, df, sim_matrix):
        self.df = df.reset_index(drop=True)
        self.sim_matrix = sim_matrix

    def search(self, query, k=5):
        matches = self.df[
            self.df["title_clean"].str.lower().str.contains(query.lower(), na=False, regex=False)
        ]
        return matches.head(k)

    def recommend_by_index(self, idx, top_k=10):
        scores = self.sim_matrix[idx]
        top_idx = np.argsort(scores)[::-1][1:top_k+1]

        results = []
        for i in top_idx:
            results.append({
                "title": self.df.iloc[i]["title_clean"],
                "score": float(scores[i])
            })

        return pd.DataFrame(results)

    def recommend(self, title, top_k=10):
        matches = self.search(title)

        if matches.empty:
            print("❌ Film non trovato")
            return None

        idx = matches.index[0]
        selected_title = matches.iloc[0]["title_clean"]

        print(f"\n🎬 Consigli per: {selected_title}\n")

        recs = self.recommend_by_index(idx, top_k)

        for _, row in recs.iterrows():
            print(f"👉 {row['title']} ({row['score']:.3f})")

        return recs